# 15 — Passive Aggressive Classifier

**Algorithm:** Passive-Aggressive Classifier (PA-I / PA-II)  
**Features:** TF-IDF (50K unigrams + bigrams)  
Passive-Aggressive algorithms are online learners that stay passive on correct predictions but update aggressively on mistakes. Well-suited to streaming text classification.

In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import (setup_paths, evaluate_and_print, plot_confusion_matrix,
                   plot_roc_curve, plot_pr_curve)
from sklearn.metrics import accuracy_score
BASE_DIR, FIGURES_DIR = setup_paths()

In [ ]:
from preprocess import load_all_datasets, get_train_test_split
df = load_all_datasets()
X_train, X_test, y_train, y_test = get_train_test_split(df)
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True, min_df=2, max_df=0.95)
X_tr = tfidf.fit_transform(X_train)
X_te = tfidf.transform(X_test)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## Training Convergence via partial_fit

In [ ]:
from sklearn.linear_model import PassiveAggressiveClassifier
classes = np.unique(y_train)
n_epochs = 20
batch_size = 2048
n = X_tr.shape[0]
tr_accs, te_accs = [], []
pa_partial = PassiveAggressiveClassifier(max_iter=1, random_state=42)
for epoch in range(1, n_epochs + 1):
    idx = np.random.permutation(n)
    for start in range(0, n, batch_size):
        batch = idx[start:start + batch_size]
        pa_partial.partial_fit(X_tr[batch], y_train[batch], classes=classes)
    tr_accs.append(accuracy_score(y_train, pa_partial.predict(X_tr)))
    te_accs.append(accuracy_score(y_test,  pa_partial.predict(X_te)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, n_epochs+1), tr_accs, 'o-', color='#3498db', lw=2, ms=5, label='Train Accuracy')
ax.plot(range(1, n_epochs+1), te_accs, 's-', color='#e74c3c', lw=2, ms=5, label='Test Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Passive Aggressive — Training Convergence')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'passive_aggressive_convergence.png'), dpi=150, bbox_inches='tight')
plt.show()

## Final Model

In [ ]:
MODEL_NAME = 'Passive Aggressive'
t0 = time.time()
model = PassiveAggressiveClassifier(max_iter=1000, random_state=42)
model.fit(X_tr, y_train)
train_time = time.time() - t0
metrics, y_pred, y_prob = evaluate_and_print(model, X_te, y_test, MODEL_NAME, train_time)

In [ ]:
plot_confusion_matrix(y_test, y_pred, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_roc_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_pr_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

## C (Aggressiveness) Sensitivity

In [ ]:
C_vals = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
tr_a, te_a = [], []
for C in C_vals:
    m = PassiveAggressiveClassifier(C=C, max_iter=200, random_state=42)
    m.fit(X_tr, y_train)
    tr_a.append(accuracy_score(y_train, m.predict(X_tr)))
    te_a.append(accuracy_score(y_test, m.predict(X_te)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(C_vals, tr_a, 'o-', color='#3498db', lw=2, ms=7, label='Train')
ax.semilogx(C_vals, te_a, 's-', color='#e74c3c', lw=2, ms=7, label='Test')
ax.set_xlabel('C (aggressiveness, log scale)')
ax.set_ylabel('Accuracy')
ax.set_title('Passive Aggressive — C Sensitivity')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'passive_aggressive_C_sensitivity.png'), dpi=150, bbox_inches='tight')
plt.show()